### Pydantic

In [34]:
import os
from dotenv import load_dotenv

load_dotenv()

True

In [35]:
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

In [36]:
from langchain_groq import ChatGroq

llm= ChatGroq(model="gemma2-9b-it")
response= llm.invoke("Hi")
response.content

'Hello! 👋  How can I help you today?\n'

In [9]:
from pydantic import BaseModel, Field

class ProductDetails(BaseModel):
    product_name: str = Field(description="Product Name")
    product_details: str = Field(description="Product Details")
    price: int = Field(description="Price in USD")

In [17]:
structured_output= llm.with_structured_output(ProductDetails)
structured_output

RunnableBinding(bound=ChatGroq(client=<groq.resources.chat.completions.Completions object at 0x000001E38E315490>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001E38E3169F0>, model_name='gemma2-9b-it', model_kwargs={}, groq_api_key=SecretStr('**********')), kwargs={'tools': [{'type': 'function', 'function': {'name': 'ProductDetails', 'description': '', 'parameters': {'properties': {'product_name': {'description': 'Product Name', 'type': 'string'}, 'product_details': {'description': 'Product Details', 'type': 'string'}, 'price': {'description': 'Price in USD', 'type': 'integer'}}, 'required': ['product_name', 'product_details', 'price'], 'type': 'object'}}}], 'ls_structured_output_format': {'kwargs': {'method': 'function_calling'}, 'schema': {'type': 'function', 'function': {'name': 'ProductDetails', 'description': '', 'parameters': {'properties': {'product_name': {'description': 'Product Name', 'type': 'string'}, 'product_details': {'description': 'Prod

In [13]:
response= structured_output.invoke("Samsung Mobile")
response

ProductDetails(product_name='Galaxy S23', product_details='A new smartphone with advanced camera features', price=500)

In [16]:
response= structured_output.invoke("i-phone")
response

ProductDetails(product_name='iPhone', product_details='Latest model with advanced features', price=800)

In [18]:
from langchain_core.prompts import ChatPromptTemplate 

In [53]:
prompt= ChatPromptTemplate.from_messages([
    ("system", "You are a Smart AI Assistant, give output as per user's input"),
    ("user", "give output as per user's input {input}"),
])

chain= prompt | structured_output
response= chain.invoke({"input": "Latest i-Phone"})
print(response)

product_name='Latest i-Phone' product_details='Latest features, premium design' price=1000


In [60]:
prompts= ChatPromptTemplate.from_messages([
    ("system", "You are a Smart AI Assistant, give output as per user's input"),
    ("user", "give output as per user's input {input}"),
])

chain= prompts | structured_output
response= chain.invoke({"input": "Latest i-Phone"})
print(response)

product_name='iPhone 15' product_details='Latest iPhone with all the bells and whistles' price=999


In [54]:
from langchain_core.prompts import PromptTemplate

In [75]:
prompts= PromptTemplate(
    template="Your are a Smart AI Assistant, give `product_name`, `product_details`, `price` as per {input} given by user. price will be in USD only",
    input_variables=["input"]
)
chain= prompts | structured_output
response= chain.invoke({"input": "Samsung Mobile"})
print(response)

product_name='Samsung Mobile' product_details='Latest Smartphone' price=800


In [76]:
prompts= PromptTemplate(
    template="Your are a Smart AI Assistant, give `product_name`, `product_details`, `price` as per {input} given by user. price will be in USD only",
    input_variables=["input"]
)
chain= prompts | structured_output
response= chain.invoke({"input": "i-phone"})
print(response)

product_name='iPhone 15 Pro' product_details='Latest iPhone model with advanced camera and processor' price=999
